In [1]:
import pandas as pd
import os

print("Starting First Innings Projection Engineering...")

# 1. Load Data
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# 2. Filter for 1st Innings Only
df_1st = deliveries[deliveries['inning'] == 1].copy()

# 3. Calculate "Match State" at every single ball
# cumulative sum of runs and wickets for each match
df_1st['current_runs'] = df_1st.groupby('match_id')['total_runs'].cumsum()

# Handle wickets (is_wicket is usually 1 or 0 in deliveries.csv)
if 'is_wicket' in df_1st.columns:
    df_1st['current_wickets'] = df_1st.groupby('match_id')['is_wicket'].cumsum()
else:
    # Fallback if the column is named differently in your specific csv (like player_dismissed)
    df_1st['is_wicket'] = df_1st['player_dismissed'].notna().astype(int)
    df_1st['current_wickets'] = df_1st.groupby('match_id')['is_wicket'].cumsum()

# Calculate exact balls bowled to get accurate decimal overs
df_1st['balls_bowled'] = df_1st.groupby('match_id').cumcount() + 1
df_1st['overs_completed'] = round(df_1st['balls_bowled'] / 6, 2)

# 4. Find the "Future" (Final Score of that innings)
final_scores = df_1st.groupby('match_id')['total_runs'].sum().reset_index()
final_scores.rename(columns={'total_runs': 'final_score'}, inplace=True)

# Merge the future score back onto every ball of the match
df_1st = df_1st.merge(final_scores, on='match_id')

# Calculate the actual runs added from this point onwards
df_1st['runs_to_add'] = df_1st['final_score'] - df_1st['current_runs']

# 5. Clean up and keep only the predictive features
projection_df = df_1st[['match_id', 'batting_team', 'bowling_team', 'overs_completed', 
                        'current_runs', 'current_wickets', 'final_score', 'runs_to_add']]

# 6. Save the Intelligence Data
os.makedirs('../data/processed', exist_ok=True)
projection_df.to_csv('../data/processed/first_innings_progression.csv', index=False)

print("✅ First Innings Projection data engineered successfully!")
print("\nSample Progression (Notice how 'runs_to_add' shrinks as overs increase):")
display(projection_df[projection_df['match_id'] == projection_df['match_id'].iloc[0]].iloc[::12].head()) # Show every 2nd over of the first match

Starting First Innings Projection Engineering...
✅ First Innings Projection data engineered successfully!

Sample Progression (Notice how 'runs_to_add' shrinks as overs increase):


,match_id,batting_team,bowling_team,overs_completed,current_runs,current_wickets,final_score,runs_to_add
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0.17,1,0,222,221
12,335982,Kolkata Knight Riders,Royal Challengers Bangalore,2.17,21,0,222,201
24,335982,Kolkata Knight Riders,Royal Challengers Bangalore,4.17,44,0,222,178
36,335982,Kolkata Knight Riders,Royal Challengers Bangalore,6.17,61,1,222,161
48,335982,Kolkata Knight Riders,Royal Challengers Bangalore,8.17,72,1,222,150
